# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is accessed through its Croissant schema URL, which provides both the metadata and record definitions for programmatic access and FAIR data practices.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` package.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Use the provided Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset object
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata information
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"RecordSets: {[x['@id'] for x in dataset.metadata.recordSets] if hasattr(dataset.metadata,'recordSets') else 'N/A'}")

## 2. Data Overview
List available record sets in the dataset and the fields in each record set using their Croissant `@id`s. This is essential for referencing objects programmatically in the next steps.

In [ ]:
# Find and print all record set @id's and their field @id's
record_sets = []
if hasattr(dataset.metadata, 'recordSets'):
    record_sets = [record_set['@id'] for record_set in dataset.metadata.recordSets]
else:
    # fallback: common croissant metadata
    if hasattr(dataset.metadata, 'recordSet'):
        rs = dataset.metadata.recordSet
        if isinstance(rs, list):
            record_sets = [r['@id'] for r in rs]
        elif isinstance(rs, dict):
            record_sets = [rs['@id']]
print(f"Found {len(record_sets)} record sets:")
for rs_id in record_sets:
    print(f"  - RecordSet @id: {rs_id}")
    rs_meta = dataset.metadata.by_id(rs_id)
    if rs_meta is not None and hasattr(rs_meta, 'field'):
        fields = rs_meta.field
        if isinstance(fields, list):
            field_ids = [f['@id'] for f in fields]
        else:
            field_ids = [fields['@id']]
        print(f"     Fields (@id): {field_ids}")

## 3. Data Extraction
Load records from all record sets as DataFrames using the Croissant `@id` for each record set. Display the list of columns for one selected record set.

In [ ]:
# Load records from each record set into DataFrames
dataframes = {}
selected_record_set = None

for rs_id in record_sets:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    if selected_record_set is None and not df.empty:
        selected_record_set = rs_id  # Use the first non-empty for demonstration

if selected_record_set is not None:
    print(f"\nAvailable columns in record set {selected_record_set}:")
    print(dataframes[selected_record_set].columns.tolist())
    dataframes[selected_record_set].head()
else:
    print("No records found in any record set!")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from the chosen record set, filter, normalize, and group data for basic exploration. All field and column references use their Croissant `@id` as per best practice.

In [ ]:
import numpy as np

# You may need to inspect which numeric columns exist:
if selected_record_set is not None:
    df = dataframes[selected_record_set]
    # Attempt to auto-detect a numeric column for demonstration
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to convert some commonly named columns
        for col in df.columns:
            if "count" in col.lower() or "mw" in col.lower() or "percent" in col.lower():
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except Exception:
                    continue
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field (column @id): {numeric_field}")
        # Example thresholding
        threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by a likely categorical field
        possible_group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            print(f"Grouped by {group_field} (mean of numeric field):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
else:
    print("No record set or data available for EDA.")

## 5. Visualization
Let us create basic plots visualizing the distribution of the selected numeric field and its relationship with a categorical grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field} in {selected_record_set}')
    plt.xlabel(numeric_field)
    plt.show()
    # Boxplot by group (if exist)
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We successfully loaded the FAIR^2 human mass spectrometry dataset using Croissant and `mlcroissant`, programmatically explored metadata and records, filtered and normalized quantitative fields, grouped by possible categorical fields, and visualized distributions. This approach ensures reproducibility and enables FAIR data handling for downstream data science applications.